In [1]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import numpy as np
import pandas as pd

In [2]:
%run 00.ipynb

c:\Users\SafinSharma\miniconda3\envs\forecasting\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Export Folder : standardized_exports
Results Folder: results
24
Datasets Found: 22
agri_medium
agri_small
energy_medium
energy_small
favorita-train_medium
favorita-train_medium_cov
favorita-train_small
favorita-train_small_cov
favorita-transaction_medium
favorita-transaction_small
m4-daily_medium
m4-daily_small
m4-hourly_medium
m4-hourly_small
m4-quarterly_medium
m4-quarterly_small
m4-weekly_medium
m4-weekly_small
m4-yearly_medium
m4-yearly_small
traffic_medium
traffic_small
5
agri_medium               rows=15343      series=246      horizon=5
5
agri_small                rows=14428      series=246      horizon=5
24
energy_medium             rows=177600     series=370      horizon=24
24
energy_small              rows=44400      series=370      horizon=24
28
favorita-train_medium     rows=1985155    series=4504     horizon=28
favorita-train_medium_cov -> ERROR: No horizon configured for favorita-train_medium_cov
28
favorita-train_small      rows=609961     series=4504     horizon=28
favo

In [3]:
RESULT_FILE = "ml_results.csv"

initialize_results_file(
    RESULT_FILE
)

completed_runs = get_completed_runs(
    RESULT_FILE
)

print(
    f"Completed Runs: {len(completed_runs)}"
)

Completed Runs: 0


In [4]:
def build_features(df):

    df = df.copy()

    for lag in [1, 2, 3, 7]:

        df[f"lag_{lag}"] = (
            df.groupby("unique_id")["y"]
            .shift(lag)
        )

    df["rolling_mean_3"] = (
        df.groupby("unique_id")["y"]
        .shift(1)
        .rolling(3)
        .mean()
        .reset_index(level=0, drop=True)
    )

    return df

In [5]:
def create_training_frame(train_df):

    feature_df = build_features(
        train_df
    )

    feature_df = feature_df.dropna()

    feature_cols = [
        c
        for c in feature_df.columns
        if c not in [
            "unique_id",
            "ds",
            "y"
        ]
    ]

    X = feature_df[feature_cols]

    y = feature_df["y"]

    return (
        X,
        y,
        feature_cols
    )

In [6]:
def generate_features_from_history(
    history
):

    return {

        "lag_1":
        history[-1],

        "lag_2":
        history[-2],

        "lag_3":
        history[-3],

        "lag_7":
        history[-7],

        "rolling_mean_3":
        np.mean(history[-3:])
    }

In [7]:
def recursive_predict(
    model,
    train_df,
    horizon
):

    forecasts = []

    for uid, group in train_df.groupby(
        "unique_id"
    ):

        history = (
            group["y"]
            .astype(float)
            .values
            .tolist()
        )

        series_preds = []

        for _ in range(horizon):

            features = (
                generate_features_from_history(
                    history
                )
            )

            pred = model.predict(
                pd.DataFrame(
                    [features]
                )
            )[0]

            series_preds.append(pred)

            history.append(pred)

        forecasts.extend(series_preds)

    return np.array(
        forecasts
    )

In [8]:
def train_lightgbm(
    X,
    y
):

    model = LGBMRegressor(

        n_estimators=100,
        learning_rate=0.05,

        random_state=RANDOM_SEED,
        verbose=-1
    )

    model.fit(
        X,
        y
    )

    return model

In [9]:
def train_xgboost(
    X,
    y
):

    model = XGBRegressor(

        n_estimators=100,
        learning_rate=0.05,
        max_depth=6,

        random_state=RANDOM_SEED,
        verbosity=0
    )

    model.fit(
        X,
        y
    )

    return model

In [10]:
MODELS = {

    "LightGBM":
    train_lightgbm,

    "XGBoost":
    train_xgboost
}   

In [11]:
def evaluate_ml_model(
    train_df,
    test_df,
    horizon,
    trainer
):

    X_train, y_train, feature_cols = (
        create_training_frame(
            train_df
        )
    )

    model = trainer(
        X_train,
        y_train
    )

    predictions = recursive_predict(
        model,
        train_df,
        horizon
    )

    actuals = (
        test_df["y"]
        .values
    )

    metrics = calculate_metrics(
        actuals,
        predictions
    )

    return metrics

In [12]:
for dataset_name in AVAILABLE_DATASETS:

    print("\n" + "=" * 80)
    print(dataset_name)

    try:

        (
            df,
            train_df,
            test_df,
            horizon
        ) = prepare_dataset(
            dataset_name
        )

        # Agri cleanup
        df = df.drop(
            columns=[
                "Area",
                "Item",
                "Element",
                "Unit"
            ],
            errors="ignore"
        )

        # Favorita cleanup
        if "onpromotion" in df.columns:

            df["onpromotion"] = (
                df["onpromotion"]
                .astype(str)
                .str.lower()
                .map({
                    "true": 1,
                    "false": 0
                })
                .fillna(0)
                .astype(int)
            )

        allowed = [
        "unique_id",
        "ds",
        "y"
        ]

        df = df[
            [c for c in df.columns if c in allowed]
        ]

        for model_name, trainer in MODELS.items():

            if (
                dataset_name,
                model_name
            ) in completed_runs:

                print(
                    f"SKIP : {model_name}"
                )

                continue

            print(
                f"RUN  : {model_name}"
            )

            start = start_timer()

            metrics = evaluate_ml_model(
                train_df,
                test_df,
                horizon,
                trainer
            )

            runtime = stop_timer(
                start
            )

            result = {

                "model":
                model_name,

                "dataset":
                dataset_name,

                "n_series":
                df["unique_id"].nunique(),

                "n_rows":
                len(df),

                "horizon":
                horizon,

                "runtime_seconds":
                runtime,

                **metrics
            }

            append_result(
                RESULT_FILE,
                result
            )

            print(
                f"MAE={metrics['MAE']:.4f}"
            )

    except Exception as e:

        print(
            f"ERROR: {e}"
        )


agri_medium
5
RUN  : LightGBM
MAE=1345459.3743
RUN  : XGBoost
MAE=1286657.1484

agri_small
5
RUN  : LightGBM
MAE=1359612.5231
RUN  : XGBoost
MAE=1329722.1926

energy_medium
24
RUN  : LightGBM
MAE=251.3174
RUN  : XGBoost
MAE=260.3539

energy_small
24
RUN  : LightGBM
MAE=250.2061
RUN  : XGBoost
MAE=249.7265

favorita-train_medium
28
RUN  : LightGBM
MAE=3.7118
RUN  : XGBoost
MAE=3.7726

favorita-train_medium_cov
ERROR: No horizon configured for favorita-train_medium_cov

favorita-train_small
28
RUN  : LightGBM
MAE=3.6314
RUN  : XGBoost
MAE=3.8705

favorita-train_small_cov
ERROR: No horizon configured for favorita-train_small_cov

favorita-transaction_medium
28
RUN  : LightGBM
MAE=113.7095
RUN  : XGBoost
MAE=126.9177

favorita-transaction_small
28
RUN  : LightGBM
MAE=114.0962
RUN  : XGBoost
MAE=115.0424

m4-daily_medium
20
RUN  : LightGBM
MAE=290.6103
RUN  : XGBoost
MAE=292.3222

m4-daily_small
20
RUN  : LightGBM
MAE=301.8101
RUN  : XGBoost
MAE=299.2280

m4-hourly_medium
20
RUN  : LightGB

In [13]:
results_df = pd.read_csv(
    RESULTS_FOLDER /
    RESULT_FILE
)

results_df.sort_values(
    ["dataset", "model"]
)

,model,dataset,n_series,n_rows,horizon,runtime_seconds,MAE,RMSE,SMAPE
0,LightGBM,agri_medium,246,15343,5,5.512362,1.345459e+06,1.193758e+07,72.657659
1,XGBoost,agri_medium,246,15343,5,4.654321,1.286657e+06,1.190936e+07,73.517225
2,LightGBM,agri_small,246,14428,5,3.158057,1.359613e+06,1.161104e+07,72.991220
3,XGBoost,agri_small,246,14428,5,5.109045,1.329722e+06,1.176519e+07,73.964951
4,LightGBM,energy_medium,370,177600,24,23.006917,2.513174e+02,1.244485e+03,59.320120
5,XGBoost,energy_medium,370,177600,24,54.584601,2.603539e+02,1.247657e+03,60.729730
6,LightGBM,energy_small,370,44400,24,37.818817,2.502061e+02,1.161488e+03,58.303284
7,XGBoost,energy_small,370,44400,24,54.708671,2.497265e+02,1.157936e+03,58.431972
8,LightGBM,favorita-train_medium,4504,1985155,28,417.355766,3.711823e+00,1.175724e+01,66.823173
9,XGBoost,favorita-train_medium,4504,1985155,28,499.925350,3.772649e+00,1.178912e+01,68.045867


In [14]:
results_df.groupby(
    "model"
)[
    [
        "MAE",
        "RMSE",
        "SMAPE",
        "runtime_seconds"
    ]
].mean().round(4)

,MAE,RMSE,SMAPE,runtime_seconds
model,,,,
LightGBM,169495.2444,1.474011e+06,45.3453,87.6497
XGBoost,163915.7643,1.481174e+06,46.5343,123.4076
